In [25]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss



In [26]:
X_train = pd.read_parquet('../data/processed/X_train.parquet', engine='fastparquet')
X_val = pd.read_parquet('../data/processed/X_val.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/processed/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/processed/y_train.parquet', engine='fastparquet').squeeze()
y_val = pd.read_parquet('../data/processed/y_val.parquet', engine='fastparquet').squeeze()
y_test = pd.read_parquet('../data/processed/y_test.parquet', engine='fastparquet').squeeze()

In [27]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(1248, 46) (1248,)
(67, 46) (67,)
(67, 46) (67,)


## Creating Logistic Regression Baseline Model

In [28]:
scalar = StandardScaler()
scalar.fit(X_train)
X_train_scaled = scalar.transform(X_train)
X_val_scaled = scalar.transform(X_val)
X_test_scaled = scalar.transform(X_test)

In [29]:
model = LogisticRegression(max_iter=1000, random_state = 0)
model.fit(X_train_scaled, y_train)
linear_predictions = model.predict(X_val_scaled)



In [30]:
# Metrics
linear_accuracy = accuracy_score(y_val, linear_predictions)
linear_logloss = log_loss(y_val, model.predict_proba(X_val_scaled))
linear_brier_score = brier_score_loss(y_val,model.predict_proba(X_val_scaled)[:,1])

linear_accuracy, linear_logloss, linear_brier_score

(0.6417910447761194, 0.6604250738440117, 0.22162293675460057)

## XGBoost

In [31]:
model_v1 = XGBClassifier(n_estimators=1000, learning_rate=0.05, early_stopping_rounds=5, max_depth=4, n_jobs=4, random_state=0)
model_v1.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
XGB_predictions = model_v1.predict(X_val)


In [32]:
XGB_accuracy = accuracy_score(y_val, XGB_predictions)
XGB_logloss = log_loss(y_val, model_v1.predict_proba(X_val))
XGB_brier_score = brier_score_loss(y_val, model_v1.predict_proba(X_val)[:,1])

XGB_accuracy, XGB_logloss, XGB_brier_score

(0.6567164179104478, 0.6078527812096003, 0.21224144101142883)